# Contextualized statement retrieval

Run from the repository root or `notebooks/`. This notebook reads all downloaded
`data/sources/*.json.gz` papers, extracts theorem-like statements, selects up to
three per paper with llama.cpp, and prepends each statement's contextualization
before embedding. Refreshed sources must contain `raw_latex`; nothing is downloaded
by the source-loading step.

Use the project's Python environment with SciPy and Jupyter installed, start your
llama.cpp server, and configure the paths below. The embedding model may download
weights on first use. Selection and contextualization can take several requests
per paper; rerunning those cells repeats the requests.

Retrieval uses exact SciPy cosine distances, with each paper scored by its best
statement pair. Both sides use the document embedding prompt for symmetric
paper-to-paper search. There is no retrieval API, database, or reranker involved.
The label-loading and metric cells are executed directly from the sibling
`retrieval_eval.ipynb` so their definitions stay in sync; its API cells are never run.

In [1]:
import ast
import gzip
import json
import re
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.spatial.distance import cdist
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd().resolve()
if not (ROOT / "paper_clustering").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "paper_clustering").is_dir():
    raise RuntimeError("Run from the repository root or notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from paper_clustering.contextualize import contextualize, select_relevant_statements
from datetime import datetime
from paper_clustering.data_models import ArxivSection, Paper, PaperMetadata
from paper_clustering.embedding import embed
from paper_clustering.extract.extract_statements import build_statement_dependency, extract_statements
from paper_clustering.llama import LlamaConnector

SOURCE_DIR = ROOT / "data" / "sources"
LABEL_DIR = Path.home() / "Downloads"
ARXIV_IDS = None  # All labelled queries, or a nonempty list; corpus always uses all sources.
K = 10  # 10–100; recall and cross-field hit use the first 10 papers.
EMBEDDING_MODEL = "nomic-ai/nomic-embed-text-v2-moe"
BATCH_SIZE = 8
DEVICE = None  # Automatically choose CUDA, MPS, or CPU.
LLAMA_SERVER_URL = None  # Uses LLAMA_CPP_SERVER_URL from the environment/.env.
RERANK = False  # Required by the shared metric reporting cell.

if not isinstance(K, int) or isinstance(K, bool) or not 10 <= K <= 100:
    raise ValueError("K must be an integer between 10 and 100.")

## Load evaluation labels

Use the same three JSONL label files and normalization as `retrieval_eval.ipynb`.
The helper locates a code cell by a top-level function definition rather than its
position. It executes the entire selected cell, including label loading or metric
reporting, in this notebook's namespace.

In [2]:
def run_evaluation_cell(function_name):
    path = ROOT / "notebooks" / "retrieval_eval.ipynb"
    notebook = json.loads(path.read_text(encoding="utf-8"))
    sources = []
    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        source = "".join(cell["source"])
        if any(isinstance(node, ast.FunctionDef) and node.name == function_name
               for node in ast.parse(source).body):
            sources.append(source)
    if len(sources) != 1:
        raise ValueError(f"Expected one cell defining {function_name} in {path}.")
    exec(compile(sources[0], str(path), "exec"), globals())


run_evaluation_cell("normalize_id")

Evaluating 29 queries; 24 have cross-field positives.


## Load refreshed sources and extract statements

All source files form the candidate corpus, including unlabelled papers. Duplicate
version-normalized IDs are rejected to avoid silently mixing paper versions.
Missing raw LaTeX stops the run before any model requests. Papers with valid sources
but no extractable statements remain visible in the extraction report.

In [3]:
source_paths = sorted(SOURCE_DIR.glob("*.json.gz"))
if not source_paths:
    raise FileNotFoundError(f"No downloaded sources found in {SOURCE_DIR}.")

papers = {}
missing_latex = []
for path in source_paths:
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        record = json.load(handle)
    paper_id = normalize_id(record["metadata"]["arxiv_id"])
    if paper_id in papers:
        raise ValueError(f"Duplicate normalized paper ID: {paper_id} ({path.name})")
    sections = [
        ArxivSection(**{**section, "text": tuple(section["text"])})
        for section in record["sections"]
    ]
    if all(not section.raw_latex.strip() for section in sections):
        missing_latex.append(path.name)
    statements = [statement for section in sections for statement in extract_statements(section)]
    metadata = PaperMetadata(**{
        **record["metadata"],
        "authors": tuple(record["metadata"]["authors"]),
        "publication_date": datetime.fromisoformat(record["metadata"]["publication_date"]),
    })
    papers[paper_id] = {
        "paper": Paper(metadata=metadata, sections=tuple(sections)),
        "sections": sections, "statements": statements,
    }
    titles.setdefault(paper_id, record["metadata"]["title"])

if missing_latex:
    print(f"Missing latex for the following sources {','.join(missing_latex)}")

extraction_report = pd.DataFrame([
    {"arxiv_id": paper_id, "sections": len(paper["sections"]),
     "extracted_statements": len(paper["statements"])}
    for paper_id, paper in papers.items()
])
display(extraction_report)
print(f"Loaded {len(papers)} papers; {sum(bool(p['statements']) for p in papers.values())} contain statements.")

Missing latex for the following sources 0806.1919.json.gz,0902.3757.json.gz,0908.1448.json.gz,1011.3234.json.gz,1107.1958.json.gz,1605.09223.json.gz,2605.13628.json.gz,math%2F0406038.json.gz


,arxiv_id,sections,extracted_statements
0,0803.0929,10,11
1,0805.4471,35,30
2,0806.1919,0,0
3,0806.4404,32,22
4,0807.1736,8,12
...,...,...,...
158,math/0605536,10,18
159,math/0606088,19,108
160,math/0612070,12,14
161,math/9909110,7,25


Loaded 163 papers; 151 contain statements.


## Select and contextualize

Before selection, remove labelled theorem statements that no other statement's
proof depends on, including isolated theorems. Use the full dependency graph once;
retain supporting theorems, other statement types, and unlabelled statements.
Theorem types are theorem, mtheorem, thm, and thrm, including starred variants.
Reuse the project's relevance selector and proof-overview contextualizer. Each
embedding input contains the one-sentence context followed by the original statement.
A failure stops the cell instead of silently removing a candidate paper. Completed
papers are kept in memory; rerunning the processing cell resumes them. Rerun the
initialization cell below to discard this state after changing sources or settings.

In [4]:
contextualized_papers = {}

In [5]:
THEOREM_TYPES = {"theorem", "mtheorem", "thm", "thrm"}
connector = LlamaConnector(server_url=LLAMA_SERVER_URL)
try:
    for paper_id, paper in papers.items():
        if paper_id in contextualized_papers:
            continue
        dependencies = build_statement_dependency(paper["statements"], paper["sections"])
        used_labels = {label for targets in dependencies.values() for label in targets}
        # Use the full graph once; keep unlabelled statements whose usage is unknown.
        candidates = [
            statement for statement in paper["statements"]
            if not (
                statement.statement_type.rstrip("*") in THEOREM_TYPES
                and statement.labels
                and statement.labels[0] not in used_labels
            )
        ]
        selected = select_relevant_statements(connector, candidates, paper["paper"])
        rows = []
        for statement in selected:
            context = contextualize(connector, statement, paper["paper"], enrichment="")
            print(context)
            rows.append({
                "arxiv_id": paper_id,
                "statement_type": statement.statement_type,
                "section_header": statement.section_header,
                "raw_latex": statement.raw_latex,
                "context": context,
                "text": f"{context}\n\n{statement.raw_latex}",
            })
        contextualized_papers[paper_id] = rows
        print(f"{paper_id}: selected {len(rows)} of {len(candidates)} candidates; "
              f"removed {len(paper['statements']) - len(candidates)} top-level theorems")
finally:
    connector.session.close()

passages = pd.DataFrame([
    row for paper_id in papers for row in contextualized_papers[paper_id]
])
if passages.empty:
    raise ValueError("No statements available to embed.")
display(passages.head())

This lemma guarantees that the approximate matrix $\tilde{Z}$, obtained via the linear solver, maintains the crucial resistance bounds necessary for the sparsification to successfully preserve the cut weights.
This lemma bounds the approximation error incurred when estimating the expected covariance matrix using a finite number of random samples, ensuring the random sampling procedure for sparsification maintains accuracy.
This lemma formally guarantees that the resulting approximated matrix $\tilde{L}$ maintains the quadratic form of the original matrix $L$ within a $(1\pm\epsilon)$ factor, validating the efficacy of the constructed sparsifier.
0803.0929: selected 3 of 9 candidates; removed 2 top-level theorems
The lemma guarantees the uniqueness of the recovered low-rank matrix solution to the optimization problem, provided the sampling operator is injective on the relevant subspace of the feasible solution.
The statement provides probabilistic bounds on the entries of a random ortho

KeyboardInterrupt: 

## Embed contextualized statements

Use the pipeline's default embedding model and existing normalized embedding helper
(2048-token maximum). Long inputs may be truncated by that helper/model.

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL, trust_remote_code=True, device=DEVICE)
embeddings = np.asarray(embed(
    passages["text"].tolist(), embedding_model,
    batch_size=BATCH_SIZE, prompt="search_document: ", device=DEVICE,
))
if (embeddings.ndim != 2 or embeddings.shape[0] != len(passages)
        or not np.isfinite(embeddings).all()
        or np.any(np.linalg.norm(embeddings, axis=1) == 0)):
    raise ValueError("Expected one finite, nonzero embedding per passage.")
paper_indices = {
    paper_id: np.asarray(indices)
    for paper_id, indices in passages.groupby("arxiv_id", sort=True).indices.items()
}
print(f"Embedded {len(passages)} statements from {len(paper_indices)} papers: {embeddings.shape}")

## Exact nearest neighbours with SciPy

Compute only one query paper's statement-to-corpus distances at a time. Rank unique
candidate papers by maximum cosine similarity across all statement pairs, excluding
the query itself before taking K. Ties are broken by arXiv ID. Keep the best pair
for inspection. Missing query sources are reported as failures; a loaded query with
no statements returns an empty successful result and therefore zero recall/hit.

In [ ]:
def nearest_papers(query_id):
    if query_id not in paper_indices:
        return {}
    query_indices = paper_indices[query_id]
    similarities = 1.0 - cdist(embeddings[query_indices], embeddings, metric="cosine")
    candidates = []
    for target_id, target_indices in paper_indices.items():
        if target_id == query_id:
            continue
        scores = similarities[:, target_indices]
        query_offset, target_offset = np.unravel_index(np.argmax(scores), scores.shape)
        pair = {
            "similarity": float(scores[query_offset, target_offset]),
            "query_passage": passages.iloc[query_indices[query_offset]]["text"],
            "target_passage": passages.iloc[target_indices[target_offset]]["text"],
            "target_title": titles.get(target_id, target_id),
        }
        candidates.append((target_id, pair))
    candidates.sort(key=lambda item: (-item[1]["similarity"], item[0]))
    return {target_id: [pair] for target_id, pair in candidates[:K]}


results = {}
for query_id in query_ids:
    started = perf_counter()
    error = None if query_id in papers else "No downloaded source for query paper."
    matches = nearest_papers(query_id) if error is None else {}
    results[query_id] = {"matches": matches, "error": error, "seconds": perf_counter() - started}
print(f"Searched {len(query_ids)} labelled queries against {len(paper_indices)} indexed papers.")

## Corpus coverage and retrieval metrics

The shared evaluation reports pooled pairwise ranking accuracy (both missing = 0.5),
macro recall@10, cross-field hit@10, and comparison coverage. Missing labelled
candidates stay in metric denominators. Missing query sources are excluded as failures;
loaded queries with no statements count as successful empty results. Corpus coverage
below distinguishes unavailable papers from retrieval misses. These are top-K,
incompletely labelled metrics, directly following `retrieval_eval.ipynb`.

In [ ]:
coverage = pd.DataFrame([
    {"arxiv_id": query_id, "source_available": query_id in papers,
     "query_indexed": query_id in paper_indices,
     "labelled_positives": len(labels[query_id]["positives"]),
     "indexed_positives": len(labels[query_id]["positives"] & paper_indices.keys()),
     "labelled_negatives": len(labels[query_id]["negatives"]),
     "indexed_negatives": len(labels[query_id]["negatives"] & paper_indices.keys())}
    for query_id in query_ids
])
display(coverage)
run_evaluation_cell("evaluate_query")

## Inspect the best statement pairs

Choose a query below to see the retrieved paper order, scores, and full contextualized
statement pairs. This cell makes no model or retrieval requests.

In [ ]:
INSPECT_QUERY = query_ids[0] if query_ids else None
if INSPECT_QUERY is not None:
    result = results[INSPECT_QUERY]
    print(f"Query: {INSPECT_QUERY} — {titles.get(INSPECT_QUERY, '')}")
    if result["error"]:
        print(result["error"])
    elif not result["matches"]:
        print("No matches: the query or candidate corpus has no indexed statements.")
    for rank, (target_id, pairs) in enumerate(result["matches"].items(), 1):
        pair = pairs[0]
        print(f"\n{rank}. {target_id} — {pair['target_title']} — cosine={pair['similarity']:.4f}")
        print("Query statement:\n" + pair["query_passage"])
        print("Neighbour statement:\n" + pair["target_passage"])